In [1]:
# Scenario: Student Course Recommendations
# Imagine you’re running an online learning platform. Students type queries like “beginner-friendly Python tutorials”, 
# but your catalog has courses titled “Introduction to Programming with Python”.
# Traditional keyword search might miss the match, but embeddings + vector search (Chroma locally or Pinecone in the cloud)
#  can connect them semantically.

# 🧩 Teaching Exercise Flow
# - Documents (Courses)
# - "Python is a high-level programming language"
# - "Machine learning models need training data"
# - "Dogs are loyal and friendly animals"
# - "Cats are independent and curious pets"
# - Embedding Model
# - Converts each course description into a vector (numbers capturing meaning).
# - Indexing
# - Chroma (local): Great for classroom demos, no API key needed.
# - Pinecone (cloud): Scales to billions of items, perfect for production.
# - Query
# - Student asks: “What animals make good companions?”
# - Embedding of query is compared against course embeddings.
# - Results
# - Top matches:
# - Cats are independent and curious pets
# - Dogs are loyal and friendly animals

In [2]:
# Install required libraries
!pip install chromadb sentence-transformers

import chromadb
from sentence_transformers import SentenceTransformer

# Step 1: Course documents (or content)
documents = [
    "Python is a high-level programming language",
    "Machine learning models need training data",
    "Dogs are loyal and friendly animals",
    "Cats are independent and curious pets"
]

# Step 2: Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Step 3: Convert documents into embeddings
embeddings = model.encode(documents)

# Step 4: Create Chroma client (local vector database)
client = chromadb.Client()

collection = client.create_collection(name="courses")

# Step 5: Store embeddings in vector database
for i, doc in enumerate(documents):
    collection.add(
        documents=[doc],
        embeddings=[embeddings[i].tolist()],
        ids=[str(i)]
    )

# Step 6: Student query
query = "What animals make good companions?"

# Convert query to embedding
query_embedding = model.encode(query).tolist()

# Step 7: Search similar vectors
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=2
)

# Step 8: Display results
print("Query:", query)
print("\nTop Matches:")

for doc in results["documents"][0]:
    print("-", doc)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query: What animals make good companions?

Top Matches:
- Dogs are loyal and friendly animals
- Cats are independent and curious pets


In [4]:
# OPTION A: Chroma (local, no API key needed)
!pip install chromadb
!pip install sentence-transformers
import chromadb
from sentence_transformers import SentenceTransformer

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Create Chroma client and collection
client = chromadb.Client()  # or chromadb.PersistentClient(path="./db")
# collection = client.create_collection("docs")

# Sample documents
documents = [
    "Python is a high-level programming language",
    "Machine learning models need training data",
    "Dogs are loyal and friendly animals",
    "Cats are independent and curious pets",
]

# Encode documents into embeddings
embeddings = model.encode(documents).tolist()

# Add to Chroma collection
collection.add(
    documents=documents,
    embeddings=embeddings,
    ids=[f"doc{i}" for i in range(len(documents))],
    metadatas=[{"source": "demo", "idx": i} for i in range(len(documents))]
)

# Query with semantic search
query = "A name of good programming language"
q_emb = model.encode([query]).tolist()
results = collection.query(query_embeddings=q_emb, n_results=3)

# Print results
for doc, dist in zip(results["documents"][0], results["distances"][0]):
    print(f"[{dist:.3f}] {doc}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[0.685] Python is a high-level programming language
[1.746] Machine learning models need training data
[1.768] Dogs are loyal and friendly animals
